# 06. Extra analyses (no existing model is retrained)
A. Paired comparisons with confidence intervals (no training)
B. Recalibrating the DP models' probabilities (no training)
C. Patient-level attack evaluation (no training; run after 05b)
D. Ordinary regularisation as a baseline (trains **new** non-private models)
E. Clipping-norm sweep at eps = 10 (trains **new** DP models)
Each section can be run on its own.

In [ ]:
import os, sys, json, gc, time, warnings, subprocess
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = os.environ.get("PA_ROOT", "/content/drive/MyDrive/privacy-audit/")
sys.path.insert(0, ROOT)          # pa_lib.py must sit in the privacy-audit folder on Drive
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip('opacus')
from pa_lib import *
make_dirs()
print(versions())


Mounted at /content/drive
{'python': '3.13.18', 'numpy': '2.1.3', 'pandas': '2.2.3', 'torch': '2.11.0+cpu', 'opacus': '1.6.0', 'art': None, 'sklearn': '1.6.1'}


In [ ]:
# A. Are the differences reliable across the 5 seeds? (paired by seed, 95% confidence interval)
ms = pd.read_csv(RESULTS + "multiseed_results.csv"); lr = pd.read_csv(RESULTS + "lira_results.csv")
def get(df, model, col):
    return df[df["model"] == model].sort_values("seed")[col].values
rows = []
for ref in ("baseline_early", "baseline_e50"):
    for m in [f"dpsgd_eps{e}" for e in EPSILONS]:
        for label, df, col in (("usefulness: ROC-AUC", ms, "test_auc"), ("usefulness: PR-AUC", ms, "test_pr_auc"),
                               ("attack score (MIA)", ms, "attack_auc"), ("attack score (LiRA)", lr, "auc"),
                               ("LiRA caught at 1% false alarms", lr, "tpr@1%fpr")):
            if m in set(df["model"]) and ref in set(df["model"]):
                d, lo, hi = paired_ci(get(df, m, col), get(df, ref, col))
                rows.append({"DP model": m, "compared with": ref, "measure": label, "difference": d, "CI low": lo, "CI high": hi,
                             "reliable?": "yes" if (lo > 0 or hi < 0) else "no (interval includes 0)"})
pc = pd.DataFrame(rows); pc.round(4).to_csv(RESULTS + "paired_comparisons.csv", index=False)
print(pc[pc["compared with"] == "baseline_early"].round(3).to_string(index=False))


    DP model  compared with                        measure  difference  CI low  CI high                reliable?
dpsgd_eps0.5 baseline_early            usefulness: ROC-AUC      -0.043  -0.053   -0.032                      yes
dpsgd_eps0.5 baseline_early             usefulness: PR-AUC      -0.023  -0.028   -0.018                      yes
dpsgd_eps0.5 baseline_early             attack score (MIA)      -0.002  -0.007    0.004 no (interval includes 0)
dpsgd_eps0.5 baseline_early            attack score (LiRA)      -0.033  -0.047   -0.020                      yes
dpsgd_eps0.5 baseline_early LiRA caught at 1% false alarms      -0.010  -0.013   -0.007                      yes
  dpsgd_eps1 baseline_early            usefulness: ROC-AUC      -0.027  -0.035   -0.019                      yes
  dpsgd_eps1 baseline_early             usefulness: PR-AUC      -0.017  -0.023   -0.011                      yes
  dpsgd_eps1 baseline_early             attack score (MIA)      -0.000  -0.012    0.012 no (inte

In [ ]:
# B. Recalibrate the DP models' probabilities (fitted on VALIDATION only, checked on test; AUC does not change)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
Xva, yva = load_split("target_val"); Xte, yte = load_split("target_test"); ytr = load_split("target_train")[1]
Z = np.load(RESULTS + "predictions.npz")          # saved by notebook 03
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)
rows = []
for name in model_names():
    for s in SEEDS:
        pv, pt = Z[f"{name}__{s}__val"], Z[f"{name}__{s}__test"]
        pc_ = LogisticRegression(C=1e6).fit(logit(pv), yva).predict_proba(logit(pt))[:, 1]
        rows.append({"model": name, "seed": s, "brier_before": brier_score_loss(yte, pt), "brier_after": brier_score_loss(yte, pc_),
                     "ece_before": ece(yte, pt), "ece_after": ece(yte, pc_), "mean_prob_after": float(pc_.mean())})
cal = pd.DataFrame(rows).groupby("model").mean(numeric_only=True).drop(columns="seed").reindex(model_names()).round(3)
cal.to_csv(RESULTS + "calibration_after_adjustment.csv"); print(cal.to_string())
print("test base rate:", round(float(yte.mean()), 3), "| the adjusted mean probability should be close to it")


                brier_before  brier_after  ece_before  ece_after  mean_prob_after
model                                                                            
baseline_e50           0.148        0.098       0.148      0.011            0.118
baseline_early         0.097        0.097       0.011      0.012            0.113
dpsgd_eps0.5           0.107        0.098       0.097      0.014            0.117
dpsgd_eps1             0.107        0.098       0.097      0.013            0.115
dpsgd_eps2             0.107        0.098       0.097      0.010            0.115
dpsgd_eps5             0.107        0.098       0.098      0.008            0.114
dpsgd_eps10            0.107        0.097       0.098      0.009            0.114
dpsgd_eps100           0.107        0.097       0.098      0.010            0.113
test base rate: 0.111 | the adjusted mean probability should be close to it


In [ ]:
# C. Patient-level evaluation: is a whole PATIENT's membership easier to see than a single visit's?
Xtr, ytr = load_split("target_train"); Xte, yte = load_split("target_test")
input_dim = Xtr.shape[1]; n_tr = len(Xtr)
pool_X = np.concatenate([Xtr, Xte]); pool_y = np.concatenate([ytr, yte])
pool_pid = np.concatenate([load_patients("target_train"), load_patients("target_test")])
N = len(pool_y); idx = np.arange(N)
member_row = np.r_[np.ones(n_tr, bool), np.zeros(N - n_tr, bool)]
def lira_set(cfg):
    fs = sorted(f for f in os.listdir(LIRA + cfg) if f.endswith(".npz"))
    d = [np.load(LIRA + cfg + "/" + f) for f in fs]
    return np.stack([x["phi"] for x in d]), np.stack([x["mask"] for x in d])
rows = []
for cfg in model_names():
    if not os.path.isdir(LIRA + cfg):
        continue
    phis, masks = lira_set(cfg)
    for seed in SEEDS:
        z = get_logits(load_model(cfg, seed, input_dim), pool_X)
        s = lira_scores(z[idx, pool_y] - z[idx, 1 - pool_y], phis, masks)
        g = patient_level(s, pool_pid, member_row)
        row = {"model": cfg, "seed": seed, "visit_auc": roc_auc_score(member_row, s),
               "patient_auc": roc_auc_score(g["member"], g["s"]),
               "patient_tpr@1%": tpr_at_fpr(g["member"].values.astype(int), g["s"].values, 0.01)}
        for nm, sel in (("one_visit", g["visits"] == 1), ("several_visits", g["visits"] >= 2)):
            gg = g[sel]
            row[f"auc_{nm}"] = roc_auc_score(gg["member"], gg["s"]) if gg["member"].nunique() == 2 else np.nan
        rows.append(row)
pl = pd.DataFrame(rows); pl.to_csv(RESULTS + "patient_level_attack.csv", index=False)
print(pl.groupby("model")[["visit_auc", "patient_auc", "patient_tpr@1%", "auc_one_visit", "auc_several_visits"]].mean().round(3).to_string())
print("share of patients with several visits (train / test):",
      round(float((patient_level(np.zeros(N), pool_pid, member_row).query("member").visits >= 2).mean()), 3),
      round(float((patient_level(np.zeros(N), pool_pid, member_row).query("not member").visits >= 2).mean()), 3))


                visit_auc  patient_auc  patient_tpr@1%  auc_one_visit  auc_several_visits
model                                                                                    
baseline_e50        0.639        0.641           0.053          0.583               0.804
baseline_early      0.534        0.526           0.016          0.506               0.590
dpsgd_eps0.5        0.501        0.502           0.013          0.499               0.513
dpsgd_eps1          0.502        0.504           0.014          0.501               0.515
dpsgd_eps10         0.511        0.510           0.015          0.505               0.528
dpsgd_eps100        0.520        0.516           0.014          0.510               0.536
dpsgd_eps2          0.506        0.508           0.013          0.504               0.520
dpsgd_eps5          0.509        0.509           0.014          0.505               0.525
share of patients with several visits (train / test): 0.243 0.228


In [ ]:
# D. Does ordinary regularisation give the same protection as DP? (new non-private models with weight decay)
from sklearn.metrics import roc_auc_score, average_precision_score
DO_REGULARISED = True
WD = [1e-3, 1e-2, 1e-1]
EXTRA = MODELS + "extra/"
if DO_REGULARISED:
    Xtr, ytr = load_split("target_train"); Xte, yte = load_split("target_test")
    input_dim = Xtr.shape[1]; n_tr = len(Xtr)
    pool_X = np.concatenate([Xtr, Xte]); pool_y = np.concatenate([ytr, yte]); N = len(pool_y); idx = np.arange(N)
    pool_pid = np.concatenate([load_patients("target_train"), load_patients("target_test")])
    labels = np.r_[np.ones(n_tr), np.zeros(N - n_tr)]
    rows = []
    for wd in WD:
        name = f"baseline_wd{wd:g}"
        os.makedirs(LIRA + name, exist_ok=True)
        for seed in SEEDS:
            if not os.path.exists(f"{EXTRA}{name}_seed{seed}.pt"):
                m, meta = train_plain(Xtr, ytr, EPOCHS, seed, weight_decay=wd)
                save_model(m, name, seed, meta, folder=EXTRA)
        for k in range(32):                                            # 32 practice models, patient masks
            path = f"{LIRA}{name}/shadow_{k}.npz"
            if not os.path.exists(path):
                mask = patient_masks(pool_pid, 0.7, np.random.RandomState(k))
                m = train_plain(pool_X[mask], pool_y[mask], EPOCHS, 500 + k, weight_decay=wd)[0]
                z = get_logits(m, pool_X)
                np.savez(path, phi=z[idx, pool_y] - z[idx, 1 - pool_y], mask=mask)
        files = [np.load(f"{LIRA}{name}/shadow_{k}.npz") for k in range(32)]
        phis = np.stack([f["phi"] for f in files]); masks = np.stack([f["mask"] for f in files])
        for seed in SEEDS:
            m = load_model(name, seed, input_dim, folder=EXTRA)
            pte = proba(m, Xte)[:, 1]
            z = get_logits(m, pool_X); s = lira_scores(z[idx, pool_y] - z[idx, 1 - pool_y], phis, masks)
            rows.append({"model": name, "seed": seed, "test_roc_auc": roc_auc_score(yte, pte), "test_pr_auc": average_precision_score(yte, pte),
                         "lira_auc": roc_auc_score(labels, s), "lira_tpr@1%": tpr_at_fpr(labels, s, 0.01)})
        print(name, "done")
    reg = pd.DataFrame(rows); reg.to_csv(RESULTS + "regularised_baselines.csv", index=False)
    summ = reg.groupby("model")[["test_roc_auc", "test_pr_auc", "lira_auc", "lira_tpr@1%"]].mean().round(3)
    ms = pd.read_csv(RESULTS + "multiseed_results.csv"); lr = pd.read_csv(RESULTS + "lira_results.csv")
    ref = ms.groupby("model")[["test_auc", "test_pr_auc"]].mean().join(lr.groupby("model")[["auc", "tpr@1%fpr"]].mean())
    ref.columns = ["test_roc_auc", "test_pr_auc", "lira_auc", "lira_tpr@1%"]
    print(pd.concat([summ, ref.loc[["baseline_e50", "baseline_early", "dpsgd_eps1", "dpsgd_eps10", "dpsgd_eps100"]].round(3)]).to_string())
    print("\nCompare rows with similar test_roc_auc: does a DP model leak less than ordinary regularisation at the same usefulness?")


baseline_wd0.001 done
baseline_wd0.01 done
baseline_wd0.1 done
                  test_roc_auc  test_pr_auc  lira_auc  lira_tpr@1%
model                                                             
baseline_wd0.001         0.560        0.137     0.635        0.084
baseline_wd0.01          0.621        0.165     0.520        0.015
baseline_wd0.1           0.582        0.151     0.509        0.021
baseline_e50             0.542        0.129     0.639        0.074
baseline_early           0.609        0.165     0.534        0.025
dpsgd_eps1               0.582        0.148     0.502        0.015
dpsgd_eps10              0.604        0.157     0.511        0.017
dpsgd_eps100             0.609        0.161     0.520        0.019

Compare rows with similar test_roc_auc: does a DP model leak less than ordinary regularisation at the same usefulness?


In [ ]:
# E. Clipping-norm sweep at eps = 10 (new DP models): does clipping explain the very low probabilities?
DO_CLIP = True
CLIPS = [0.1, 0.5, 1.0, 5.0, 10.0]
EXTRA = MODELS + "extra/"
if DO_CLIP:
    Xtr, ytr = load_split("target_train"); Xte, yte = load_split("target_test")
    input_dim = Xtr.shape[1]
    rows = []
    for c in CLIPS:
        for seed in SEEDS:
            if c == 1.0:
                m = load_model("dpsgd_eps10", seed, input_dim)           # the existing model, not retrained
            else:
                name = f"dpsgd_eps10_clip{c:g}"
                if not os.path.exists(f"{EXTRA}{name}_seed{seed}.pt"):
                    m, meta = train_dp(Xtr, ytr, 10, seed, clip=c)
                    save_model(m, name, seed, meta, folder=EXTRA)
                m = load_model(name, seed, input_dim, folder=EXTRA)
            p = proba(m, Xte)[:, 1]
            rows.append({"clip": c, "seed": seed, "test_roc_auc": roc_auc_score(yte, p), "mean_probability": float(p.mean()),
                         "share_over_0.5": float((p >= 0.5).mean()), "ece": ece(yte, p)})
        print("clip", c, "done")
    cl = pd.DataFrame(rows); cl.to_csv(RESULTS + "clipping_sweep.csv", index=False)
    print(cl.groupby("clip")[["test_roc_auc", "mean_probability", "share_over_0.5", "ece"]].mean().round(3).to_string())
    print("test base rate:", round(float(yte.mean()), 3), "| if the mean probability rises toward it as the clip grows, clipping is the cause")


clip 0.1 done
clip 0.5 done
clip 1.0 done
clip 5.0 done
clip 10.0 done
      test_roc_auc  mean_probability  share_over_0.5    ece
clip                                                       
0.1          0.589             0.001             0.0  0.110
0.5          0.599             0.006             0.0  0.105
1.0          0.604             0.013             0.0  0.098
5.0          0.614             0.090             0.0  0.025
10.0         0.612             0.107             0.0  0.008
test base rate: 0.111 | if the mean probability rises toward it as the clip grows, clipping is the cause
